# RNN 
- 2 Entradas PwmD, PwmE
- 1 Saida $\Delta \theta $ 
- 1 Camada
- n neurônios

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers

Datasets = []
PREDICTORS = ["PwmD", "PwmE", "Theta"]   
TARGET_INT = ["X"]  
TARGET = ["DeltaX"]
     
TIME_STEPS = 3
TS = 0.07

In [13]:
for i in range(4):
    Dataset = pd.read_excel(f"../../../../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(2):   
    Dataset = pd.read_csv(f"../../../../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [14]:
NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train1, Train2], ignore_index=True)

for i in range(4):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [15]:
print(OUT_SCALER.mean_, OUT_SCALER.scale_)

[0.00014637] [0.09065103]


In [16]:
NormDatasets[0]


,Unnamed: 0,X,Y,Theta,Wd,We,WdRef,WeRef,PwmD,PwmE,DeltaX
0,0.00,0.00,0.70,-0.032459,0.00,0.0,0.00,0.00,0.221564,0.111917,-0.001615
1,0.07,0.00,0.70,-0.032459,0.00,0.0,3.28,3.28,0.221564,0.111917,-0.001615
2,0.14,0.00,0.70,-0.032459,0.00,0.0,3.28,3.28,0.803114,0.594038,-0.001615
3,0.21,0.00,0.70,-0.032459,0.06,0.0,3.28,3.28,0.803114,0.594038,-0.001615
4,0.28,0.00,0.70,-0.011465,0.33,0.0,3.28,3.28,1.025343,0.786494,-0.001615
...,...,...,...,...,...,...,...,...,...,...,...
971,67.97,0.01,0.69,-1.711986,0.00,0.0,0.85,2.03,-0.400134,-0.179313,-0.001615
972,68.04,0.01,0.69,-1.711986,0.00,0.0,0.85,2.03,-0.340030,-0.062625,-0.001615
973,68.11,0.01,0.69,-1.711986,0.00,0.0,0.85,2.03,-0.340030,-0.062625,-0.001615
974,68.18,0.01,0.69,-1.711986,0.00,0.0,0.85,2.03,-0.279927,0.056412,-0.001615


In [17]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

In [18]:
x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (1949, 3, 3)
Dimensão da saida: (1949, 1)
Dimensão da entrada: (1268, 3, 3)
Dimensão da saida: (1268, 1)


In [19]:
import matplotlib.pyplot as plt

def PlotHistory(history):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True)
    plt.show()

In [20]:
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

TITLES = ["train1", "train2", "test1", "test2", "test3", "val"]

def PlotOut(ax, title, target_name, y_true, y_pred):
    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    ax.scatter(time, y_true, marker='o', s=12, label='Amostras Reais', alpha=0.7)
    ax.scatter(time, y_pred, marker='x', s=12, label='Valores Preditos', alpha=0.7)
    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    n_targets = len(TARGET)

    metrics = {name: {"R2_train1": [], "R2_train2": [],
                      "R2_test1": [], "R2_test2": [],
                      "R2_test3": [],"R2_val": [],
                      "MSE_train1": [], "MSE_train2": [],
                      "MSE_test1": [], "MSE_test2": [],
                      "MSE_test3": [], "MSE_val": [],} for name in TARGET_INT}

    for i, dataset in enumerate(NormDatasets):
        x = dataset[PREDICTORS]
        y = dataset[TARGET]
        x, y = CreateSequences(x, y, TIME_STEPS)

        y_pred_diff = OUT_SCALER.inverse_transform(model.predict(x))
        y_diff = OUT_SCALER.inverse_transform(y)
        
        y_true = np.zeros_like(y_diff)
        y_pred = np.zeros_like(y_pred_diff)

        # valores iniciais reais (não normalizados)
        init_vals = np.array([
            Datasets[i]["Theta"].iloc[0],
            Datasets[i]["X"].iloc[0],
            Datasets[i]["Y"].iloc[0]
        ])

        # reconstrução cumulativa saída a saída
        for j in range(n_targets):
            y_true[:, j] = init_vals[j] + np.cumsum(y_diff[:, j] * TS)
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j] * TS)


        # Calcula métricas por saída
        for j, name in enumerate(TARGET_INT):
            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])
            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(f"{name} | {TITLES[i]} -> R² = {r2:.4f}, MSE = {mse:.4e}")
            
    # Retorna métricas médias para análise posterior
    return metrics

In [21]:
import numpy as np
import pandas as pd
from tensorflow import keras
from keras.callbacks import EarlyStopping
from keras import initializers
from keras import regularizers
from itertools import product

# tamanho da entrada
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET) 
N_MODELS = 5  # número de inicializações

seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

# arquiteturas das camadas ocultas
architectures = [
    [32], 
    [8, 4], 
    [16, 8],
    [32, 16],
    [64, 32],
    [128, 64],
    [128],
]
r_values = [0.01]

results = {}
excel_file = "resultados.xlsx"

for arch,  r in product(architectures, r_values):
    for i, s in enumerate(seeds):

        initializer = initializers.RandomNormal(seed=int(s))
        regularizer = regularizers.l2(r)

        model = keras.models.Sequential()
        model.add(keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

        # adiciona camadas RNN
        for j, n in enumerate(arch):

            # se não for a última camada RNN
            if j < len(arch) - 1:
                model.add(
                    keras.layers.SimpleRNN(
                        n,
                        activation="tanh",
                        return_sequences=True,
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )
            else:
                model.add(
                    keras.layers.SimpleRNN(
                        n,
                        activation="tanh",
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        # camada de saída
        model.add(
            keras.layers.Dense(
                OUTPUT_SIZE,
                activation="linear",
                kernel_initializer=initializer,
                kernel_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )
        # salva pesos iniciais
        w0 = model.get_weights()

        # compila
        model.compile(loss="mean_squared_error", optimizer="adam")
        early_stopping_monitor = EarlyStopping(
            monitor='val_loss',
            patience=50,
            restore_best_weights=True
        )

        history = model.fit(
            x_train, 
            y_train, 
            epochs=1000,
            callbacks=[early_stopping_monitor],
            validation_data=(x_val, y_val),
            verbose=0
        )
        
    
        model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_seed{s}"
        weights_path = f"weights/{model_name}.weights.h5"

        # salva pesos finais
        wf = model.get_weights()

        # salva pesos em arquivo
        model.save_weights(weights_path)
        model.save(f"./model/{model_name}.keras")
        
        # avalia o modelo
        metrics = EvalModel(model)

        row = {
            "model": model_name,
            "Neurons": arch,
            "weights_file": weights_path,
        }

       
        for name in TARGET_INT:
            row.update({
                f"R2_Train_1_{name}": metrics[name]["R2_train1"],
                f"MSE_Train_1_{name}": metrics[name]["MSE_train1"],
                f"R2_Train_2_{name}": metrics[name]["R2_train2"],
                f"MSE_Train_2_{name}": metrics[name]["MSE_train2"],
                f"R2_Val_{name}": metrics[name]["R2_val"],
                f"MSE_Val_{name}": metrics[name]["MSE_val"],
                f"R2_Test_1_{name}": metrics[name]["R2_test1"],
                f"MSE_Test_1_{name}": metrics[name]["MSE_test1"],
                f"R2_Test_2_{name}": metrics[name]["R2_test2"],
                f"MSE_Test_2_{name}": metrics[name]["MSE_test2"],
                f"R2_Test_3_{name}": metrics[name]["R2_test3"],
                f"MSE_Test_3_{name}": metrics[name]["MSE_test3"],
            })
        
        df = pd.DataFrame([row])

        # salva/atualiza Excel incrementalmente
        try:
            # tenta abrir existente e adicionar linha
            old = pd.read_excel(excel_file)
            new_df = pd.concat([old, df], ignore_index=True)
            new_df.to_excel(excel_file, index=False)
        except FileNotFoundError:
            # se não existir, cria arquivo novo
            df.to_excel(excel_file, index=False)

        print(f"Modelo {i} treinado e salvo no Excel")

31/31 [==============================] - 0s 2ms/step
X | train1 -> R² = 0.9681, MSE = 3.4409e-03
31/31 [==============================] - 0s 2ms/step
X | train2 -> R² = 0.9907, MSE = 7.8063e-04
31/31 [==============================] - 0s 2ms/step
X | test1 -> R² = 0.9707, MSE = 3.0702e-03
31/31 [==============================] - 0s 2ms/step
X | test2 -> R² = 0.9924, MSE = 6.2651e-04
45/45 [==============================] - 0s 1ms/step
X | test3 -> R² = -0.7050, MSE = 1.5687e-01
40/40 [==============================] - 0s 2ms/step
X | val -> R² = 0.7403, MSE = 2.4830e-02
Modelo 0 treinado e salvo no Excel
31/31 [==============================] - 0s 2ms/step
X | train1 -> R² = 0.9616, MSE = 4.1409e-03
31/31 [==============================] - 0s 2ms/step
X | train2 -> R² = 0.7026, MSE = 2.5056e-02
31/31 [==============================] - 0s 2ms/step
X | test1 -> R² = 0.9784, MSE = 2.2581e-03
31/31 [==============================] - 0s 2ms/step
X | test2 -> R² = 0.6705, MSE = 2.7113e-02
45